# [실습 4] Hugging Face Hub 배포 패키지 구성

## 실습 개요

> "로컬 학습 결과를 Hub에서 재사용 가능한 자산으로 만들려면 어떤 파일이 필요할까?"

이번 실습에서는 실제 Hub 업로드를 수행하지 않고, 업로드 직전의 로컬 저장소 구조를 만듭니다. 모델 파일, 토크나이저 파일, 설정 파일, 모델 카드 초안을 한 폴더에 모아 `from_pretrained()`로 다시 읽을 수 있는 형태를 확인합니다.

Hub 배포는 단순히 파일을 올리는 작업이 아닙니다. 라이선스, 데이터 출처, 사용 제한, 모델 카드, 공개 범위를 함께 정리해야 이후 협업과 재사용이 안정적으로 이어집니다.

## 실습 목표

1. 작은 Transformers 모델과 토크나이저를 로컬 저장소 구조로 저장한다.
2. 모델 카드 템플릿을 읽고 배포 정보를 채운다.
3. Hub 저장소에 포함될 파일 목록을 점검한다.
4. push_to_hub 전에 확인해야 할 공개 범위와 토큰 권한을 정리한다.
5. [TODO] 코드를 완성해 README 모델 카드를 생성한다.

## 데이터 출처

- 모델: `hf-internal-testing/tiny-random-bert`
- 템플릿: `templates/model_card_template.md`

---
## 1. 배포 패키지 설정 읽기

### Hub 업로드 전에 로컬 구조부터 확인하기

`push_to_hub()`는 편리하지만, 업로드 직전에 어떤 파일이 만들어지는지 모르면 재사용 가능한 저장소를 만들기 어렵습니다. 먼저 로컬 폴더에 모델과 토크나이저를 저장해 표준 구조를 확인합니다.

설정 파일에는 저장소 이름, 공개 범위, 라이선스, 데이터 출처 같은 배포 메타데이터를 담았습니다. 이런 값은 단순 설명이 아니라 모델을 어디까지 공유할 수 있는지, 다른 사람이 어떤 조건으로 사용할 수 있는지 판단하는 기준이 됩니다.

In [ ]:
import json
from pathlib import Path

from transformers import AutoModelForSequenceClassification, AutoTokenizer

metadata = json.loads(Path('config/hub_metadata.json').read_text(encoding='utf-8'))
output_dir = Path(metadata['local_repo_dir'])
output_dir.mkdir(exist_ok=True)
print('Hub 배포 메타데이터:', metadata)

---
## 2. 모델과 토크나이저 저장

### save_pretrained가 만드는 표준 파일 구조

Transformers 모델과 토크나이저는 `save_pretrained()`로 저장하면 다시 `from_pretrained()`로 불러올 수 있는 구조가 됩니다. 이 구조가 Hub에 올라가면 다른 사용자는 로컬 경로 대신 저장소 이름만 넣어 같은 인터페이스로 모델을 재사용할 수 있습니다.

모델 가중치, config, tokenizer 관련 파일이 함께 있어야 추론과 추가 학습이 안정적으로 이어집니다. 분류 라벨 수를 바꾸는 경우에는 기존 checkpoint의 head 크기와 새 설정이 다를 수 있으므로, backbone은 재사용하고 head는 새로 초기화하도록 mismatch를 허용합니다. 이 옵션이 왜 필요한지 알고 있어야 저장 로그에 warning이 나와도 당황하지 않고 해석할 수 있습니다.

In [ ]:
checkpoint = metadata['base_checkpoint']
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=metadata['num_labels'],
    ignore_mismatched_sizes=True,
)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

for path in sorted(output_dir.iterdir()):
    print('저장된 파일:', path.name)

### 저장된 config 내용 확인하기

`config.json`은 Hub에서 모델을 다시 구성할 때 가장 먼저 읽히는 파일 중 하나입니다. 어떤 모델 아키텍처를 쓸지, 라벨 수가 몇 개인지, id-label 매핑이 어떻게 되는지 같은 정보가 여기에 들어갑니다.

저장 직후 핵심 값을 확인하면 분류 head 설정이 의도대로 반영되었는지 빠르게 볼 수 있습니다. 파일이 존재하는지만 보는 것보다, 실제 내용이 실습 설정과 맞는지 확인하는 편이 안전합니다.

In [ ]:
saved_config = json.loads((output_dir / 'config.json').read_text(encoding='utf-8'))
print('저장된 model_type:', saved_config.get('model_type'))
print('저장된 num_labels:', saved_config.get('num_labels'))
print('저장된 architectures:', saved_config.get('architectures'))

---
## 3. [TODO] 모델 카드 생성

### [TODO] 모델 카드는 배포된 모델의 사용 설명서

모델 카드는 Hub 저장소를 방문한 사람이 가장 먼저 읽는 문서입니다. 모델이 어떤 데이터와 목적을 기준으로 만들어졌는지, 어떤 제한이 있는지, 라이선스는 무엇인지 적혀 있어야 재사용자가 잘못된 상황에 모델을 적용하는 일을 줄일 수 있습니다.

[TODO]에서는 템플릿 파일을 읽고 메타데이터 값으로 placeholder를 채워 `README.md`를 생성합니다. 실제 Hub에서는 이 파일이 모델 페이지의 본문으로 표시됩니다. 내부 공유 모델이라도 모델 카드가 있으면 나중에 실험 맥락과 사용 조건을 훨씬 쉽게 파악할 수 있습니다.

완성해야 할 변수는 다음과 같습니다.

- `template_values`: 템플릿의 placeholder를 채울 값으로, 여기서는 `metadata`를 사용합니다.
- `readme_text`: `template.format(**template_values)`로 생성한 모델 카드 본문입니다.
- `readme_path`: `output_dir / 'README.md'` 경로입니다.
- `write_result`: `readme_path.write_text(readme_text, encoding='utf-8')`의 반환값입니다.

In [ ]:
template = Path('templates/model_card_template.md').read_text(encoding='utf-8')

# [TODO 1] 템플릿의 placeholder를 metadata 값으로 채워 README 내용을 만드세요.
# 힌트: str.format(**template_values)를 사용할 수 있습니다.
template_values = None
readme_text = None

# [TODO 2] output_dir / 'README.md'에 readme_text를 저장하세요.
readme_path = None
write_result = None

print('생성된 모델 카드 미리보기:')
print(readme_text[:500])

---
## 4. 배포 전 파일 목록과 정책 확인

### 공개 범위와 토큰 권한 점검하기

Hub에 올릴 준비가 된 폴더는 파일 구조뿐 아니라 공개 범위와 권한도 함께 확인해야 합니다. Public 저장소는 공유와 재사용에 유리하지만, 데이터 출처와 라이선스 문제가 더 중요해집니다. Private 저장소는 내부 실험을 보호하는 데 적합하지만, 협업자 권한 관리가 필요합니다.

write 권한 토큰은 업로드에 필요하지만 노트북이나 저장소에 직접 남기면 안 됩니다. 실습에서는 실제 업로드를 하지 않고 체크리스트만 출력합니다. 이 단계의 목적은 모델을 올리기 전에 사람이 확인해야 할 항목을 코드 흐름 안에 자연스럽게 넣는 것입니다.

In [ ]:
required_files = ['config.json', 'README.md']
existing = {path.name for path in output_dir.iterdir()}
missing = [name for name in required_files if name not in existing]

print('로컬 저장소 경로:', output_dir)
print('누락된 필수 파일:', missing)
print('권장 공개 범위:', metadata['visibility'])
print('필요한 토큰 권한:', 'write' if metadata['will_push_to_hub'] else '로컬 검증만 하므로 read 권한이면 충분')

### Hub 저장소에 포함될 파일 확인하기

필수 파일 검사가 끝났다면 실제 저장소 폴더에 어떤 파일이 들어 있는지 확인합니다. `config.json`, 모델 가중치, 토크나이저 파일, `README.md`가 함께 있어야 표준 Transformers 인터페이스로 재사용하기 쉽습니다.

파일 목록 출력은 단순해 보이지만 배포 직전에는 효과적인 점검 방법입니다. 누락된 파일이 있으면 업로드 이후에 다른 사용자가 모델을 불러오는 단계에서 오류를 만나게 됩니다.

In [ ]:
print('저장소 파일 목록:')
for path in sorted(output_dir.iterdir()):
    print('-', path.name)

### 로컬 저장소에서 다시 불러오기

Hub에 올리기 전, 로컬 폴더만으로 `from_pretrained()`가 정상 동작하는지 확인합니다. 이 검증이 통과하면 Hub에 업로드된 뒤에도 같은 파일 구조로 재사용될 가능성이 높습니다.

모델과 토크나이저를 모두 다시 읽는 이유는 둘 중 하나만 정상이어도 전체 추론 파이프라인은 실패할 수 있기 때문입니다. 예를 들어 모델 파일은 맞지만 tokenizer 설정이 빠지면 입력 전처리에서 문제가 생깁니다. 배포 패키지는 저장이 아니라 재로드까지 확인해야 완성됐다고 볼 수 있습니다.

In [ ]:
reloaded_model = AutoModelForSequenceClassification.from_pretrained(output_dir)
reloaded_tokenizer = AutoTokenizer.from_pretrained(output_dir)

print('다시 불러온 모델 클래스:', type(reloaded_model).__name__)
print('다시 불러온 토크나이저 클래스:', type(reloaded_tokenizer).__name__)
print('다시 불러온 모델 label 수:', reloaded_model.config.num_labels)

### 모델 카드 필수 항목 점검하기

모델 카드는 사람이 읽는 문서이므로 파일 존재만 확인해서는 부족합니다. 최소한 모델 요약, 데이터 설명, 사용 목적, 제한 사항이 들어 있어야 재사용자가 모델의 의도와 한계를 이해할 수 있습니다.

여기서는 주요 섹션 제목이 포함되어 있는지 간단히 검사합니다. 실제 배포 전에는 각 섹션의 내용이 프로젝트 정책과 맞는지, 라이선스 표현이 정확한지, 민감한 내부 정보가 들어가지 않았는지도 함께 검토해야 합니다.

In [ ]:
readme_content = (output_dir / 'README.md').read_text(encoding='utf-8')
required_sections = ['## Model Summary', '## Dataset', '## Intended Use', '## Limitations']
for section in required_sections:
    print(f'{section} 포함 여부:', section in readme_content)